# Lecture 16 - Binary search trees and kd-trees

**Lecture Topic**: A structure where each step throws away half the data, why its cost depends entirely on its shape, and how the same idea finds nearest neighbours in space.

## Announcements

- Module C continues.
- Keep going on your project work.
- ClassBuzz poll for today: `XXXX`

## Recap: last time

- **Stack** (LIFO) and **queue** (FIFO), and why a list is a poor queue.
- The **call stack**, and reading a traceback bottom-up.
- **Recursion**: a frame per call, a depth limit, and the iterative rewrite.

Today: the **binary search tree**, where recursion is exactly the right tool and the depth stays around `log n`, as long as the tree is balanced.

Today:
 - The binary search tree and its property
 - Search cost is the height of the tree
 - Degeneracy on sorted input, and balanced trees
 - Binary search on a sorted array (the same idea, no tree)
 - kd-trees: nearest-neighbour search in space

# The binary search tree

A **binary tree** is made of nodes, each with up to two children (a left and a right). A **binary search tree** adds one rule, the BST property:

> For every node, everything in its **left** subtree is **smaller**, and everything in its **right** subtree is **larger**.

That one rule means each comparison tells you which half to keep and which half to throw away, so searching skips about half the remaining nodes at every step.

In [ ]:
class BSTNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None

    def add(self, value):
        if self.root is None:
            self.root = BSTNode(value)
        else:
            self._add(self.root, value)

    def _add(self, node, value):          # recursion: go left if smaller, else right
        if value < node.value:
            if node.left is None:
                node.left = BSTNode(value)
            else:
                self._add(node.left, value)
        else:
            if node.right is None:
                node.right = BSTNode(value)
            else:
                self._add(node.right, value)

    def contains(self, value):            # follow one path down
        node = self.root
        while node is not None:
            if value == node.value:
                return True
            node = node.left if value < node.value else node.right
        return False

    def height(self):
        def h(node):
            if node is None:
                return 0
            return 1 + max(h(node.left), h(node.right))
        return h(self.root)

In [ ]:
import random
random.seed(0)

tree = BST()
inserted = random.sample(range(1000), 200)   # 200 values in random order
for v in inserted:
    tree.add(v)

print("contains an inserted value?", tree.contains(inserted[0]))
print("contains -1?               ", tree.contains(-1))
print("height with 200 random values:", tree.height())

### Question 1

For a value stored in a BST, what are the **best-case** and **worst-case** costs of `contains`?

 - A. O(1), O(1)
 - B. O(1), O(log n)
 - C. O(log n), O(n)
 - D. O(n log n), O(n log n)
 - E. None of the above

**C, O(log n) and O(n).** `contains` follows a single path from the root down, so its cost is the **height** of the tree. A nicely balanced tree has height about `log n`; a badly shaped one can have height `n`. The next slide is where that worst case comes from.

# Search cost is the height of the tree

The 200 random values gave a height of about 15 to 20, a small multiple of `log2(200) ~ 8` (random trees are not perfectly balanced). So search is effectively `O(log n)`.

But the shape depends entirely on the **insertion order**, and one order is a disaster: already-sorted data.

In [ ]:
import math

sorted_tree = BST()
for v in range(200):            # inserted in sorted order
    sorted_tree.add(v)

random_tree = BST()
for v in random.sample(range(200), 200):
    random_tree.add(v)

print(f"log2(200)              ~ {math.log2(200):.1f}")
print(f"height, random order   = {random_tree.height()}")
print(f"height, sorted order   = {sorted_tree.height()}")

### Question 2

You build a BST by inserting values that are **already sorted** (`1, 2, 3, ...`). What is the height of the resulting tree, and so the cost of `contains`?

 - A. About `log n`; O(log n)
 - B. About `n`; O(n)
 - C. Zero; O(1)
 - D. It depends on the values, not the order

**B.** Each new value is larger than everything so far, so it always goes right. The tree degenerates into a straight line, a linked list in disguise, with height `n`. Search is back to O(n), the very thing the tree was supposed to avoid.

## Why balanced trees and database indexes exist

A plain BST is at the mercy of its input order, and real data is often sorted or nearly sorted, exactly the bad case. So nobody ships a plain BST. Real systems use **self-balancing** trees (red-black trees, AVL trees) that rearrange themselves on insert to keep the height at `O(log n)` no matter the order.

Databases, and the indexes behind a pandas join, use a **B-tree**, a shallow, wide balanced tree tuned so each step reads one disk or memory block. The guarantee is the same: lookups stay `O(log n)`, whatever order the data arrived in.

# Traversal: visiting every node

To visit every node you recurse into the left subtree, visit the node, and recurse into the right. This is **in-order** traversal, and because of the BST property it produces the values in **sorted order** for free.

In [ ]:
def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)       # everything smaller, first
    out.append(node.value)        # then this node
    inorder(node.right, out)      # then everything larger

vals = []
inorder(random_tree.root, vals)
print("first 15 in-order:", vals[:15])
print("is it sorted?    ", vals == sorted(vals))

### Question 3

What order does an **in-order** traversal of a binary search tree visit the values in?

 - A. The order they were inserted
 - B. Sorted (ascending) order
 - C. Largest to smallest
 - D. A random order

**B, sorted ascending.** Left subtree (all smaller), then the node, then the right subtree (all larger), applied recursively, yields the values in order. Here recursion is the natural fit: the depth is the tree height, about `log n`, so there is no danger of the recursion limit from last time.

## Pre-order and post-order

Visit the node at a different point and you get a different order. These come up when you serialise a tree (pre-order) or free it bottom-up (post-order).

 - **Pre-order:** node, then left, then right.
 - **In-order:** left, then node, then right (the sorted one above).
 - **Post-order:** left, then right, then node.

In [ ]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.value)
    preorder(node.left, out)
    preorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.value)

small = BST()
for v in [5, 3, 8, 2, 4, 7]:
    small.add(v)
for name, fn in [("pre-order ", preorder), ("in-order  ", inorder), ("post-order", postorder)]:
    out = []
    fn(small.root, out)
    print(name, out)

### Question 4

Which of these is a valid binary search tree?

```
       A              B              C
       5              5              5
      / \            / \            / \
     3   8          3   8          3   8
      \              \                /
       6              4              4
```

 - A
 - B
 - C
 - More than one

**B.** The rule is that *everything* in a node's left subtree must be smaller than it, not just its immediate child. In A, the 6 sits in 5's left subtree but is larger than 5. In C, the 4 sits in 5's right subtree but is smaller than 5. Only B keeps every value on the correct side of every ancestor.

# Removing a node

Removal is the operation with cases worth thinking about, so it is worth seeing once. Find the value, then handle three situations:

 - **A leaf:** detach it.
 - **One child:** replace the node with its single child.
 - **Two children:** replace the node's value with its **in-order successor** (the smallest value in its right subtree), then remove that successor, which is now an easier case.

The successor is the next value in the sorted order from the traversal above, so the tree stays ordered.

In [ ]:
def tree_min(node):
    while node.left is not None:
        node = node.left
    return node

def remove(node, value):
    if node is None:
        return None
    if value < node.value:
        node.left = remove(node.left, value)
    elif value > node.value:
        node.right = remove(node.right, value)
    else:                                   # found the node to remove
        if node.left is None:               # leaf, or a right child only
            return node.right
        if node.right is None:              # a left child only
            return node.left
        succ = tree_min(node.right)         # two children: in-order successor
        node.value = succ.value             # copy it up
        node.right = remove(node.right, succ.value)   # then delete the successor
    return node

t = BST()
for v in [5, 3, 8, 2, 4, 7, 9, 1]:
    t.add(v)
shown = []
inorder(t.root, shown)
print("start:                 ", shown)
for target, kind in [(4, "a leaf"), (8, "two children"), (2, "one child")]:
    t.root = remove(t.root, target)
    out = []
    inorder(t.root, out)
    print(f"remove {target} ({kind:>13}):", out)
    assert out == sorted(out)              # still a valid BST

In-order stays sorted after every removal, which is the check that the tree is still a valid BST. We show `remove` rather than ask you to write it; the point is to recognise the three cases and know the two-children case leans on the successor.

# The array analogue: binary search

You do not always need to build a tree to get the halving. If your data is already a **sorted array**, you can binary-search it directly: look at the middle, and since the halves are ordered, one comparison tells you which half to keep. That is O(log n) with no tree and no pointers, just indices.

In [ ]:
import bisect
import math

data = list(range(0, 1_000_000, 2))     # a sorted array: 0, 2, 4, ...

def contains_sorted(arr, x):
    i = bisect.bisect_left(arr, x)
    return i < len(arr) and arr[i] == x

print("contains 500_000?", contains_sorted(data, 500_000))
print("contains 500_001?", contains_sorted(data, 500_001))
print(f"{len(data):,} sorted items, found in about log2 = {math.log2(len(data)):.0f} steps")

So "keep it sorted" is itself a performance decision: a sorted array gives you O(log n) lookups from `bisect`, and it is why a sorted index in pandas can use fast searches that an unsorted one cannot. The tree and the sorted array are two shapes of the same `log n` halving idea.

# kd-trees: binary search in space

A BST splits a line of numbers. A **kd-tree** splits *space*, so you can ask "what is the nearest point to here?" without checking every point.

The idea: cycle through the dimensions as you go down. One level splits on `x` (is the query's `x` less than this node's?), the next on `y`, the next on `x` again, and so on. Each step rules out a whole region, so nearest-neighbour search becomes about `O(log n)` on average instead of scanning all `n` points.

In [ ]:
import numpy as np
from scipy.spatial import KDTree
import time

rng = np.random.default_rng(0)
points = rng.random((200_000, 2))        # 200k points in the unit square
queries = rng.random((2_000, 2))         # 2000 "nearest point to here?" questions

# brute force: for each query, measure the distance to every point
start = time.perf_counter()
brute = np.array([np.argmin(((points - q) ** 2).sum(axis=1)) for q in queries])
t_brute = time.perf_counter() - start

# kd-tree: build once, then answer all the queries
start = time.perf_counter()
tree = KDTree(points)
_, kd = tree.query(queries)
t_kd = time.perf_counter() - start

print(f"brute force, 2000 queries: {t_brute:.2f} s")
print(f"kd-tree (build + query):   {t_kd:.3f} s")
print(f"same answers? {np.array_equal(brute, kd)};  about {t_brute / t_kd:.0f}x faster")

Same answers for all 2000 queries, in a fraction of the time even counting the cost of building the tree, because each query skips whole regions instead of measuring every distance. This is how spatial lookups like "nearest weather station" or "which taxi zone contains this point" stay fast on large point sets. `scipy.spatial.KDTree` is the one you would actually reach for; you build it once and the win grows with every query you run against it.

## A verification trap: distance on a sphere

A kd-tree finds the nearest point by whatever distance you give it, and that is where agent code often goes wrong. A common bug is measuring distance on latitude and longitude as if they were a flat `(x, y)` plane. That breaks near the poles, where the longitude lines crowd together, and at the dateline, where longitude jumps from +180 to -180.

Here is a nearest-neighbour query where the naive distance and the real one disagree.

In [ ]:
import numpy as np

query = np.array([71.0, 179.5])          # (lat, lon) in degrees: far north, near the dateline
candidates = {
    "A: just across the dateline": np.array([71.0, -179.5]),
    "B: same side, 2 deg south":   np.array([69.0, 179.5]),
}

def flat_degrees(p, q):                   # naive: treat lat/lon as a flat plane
    return float(np.sqrt(((p - q) ** 2).sum()))

def great_circle_km(p, q):                # true distance over the sphere
    R = 6371.0
    lat1, lon1, lat2, lon2 = np.radians([p[0], p[1], q[0], q[1]])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return float(2 * R * np.arcsin(np.sqrt(a)))

for name, c in candidates.items():
    print(f"{name:28s} flat(deg)={flat_degrees(query, c):6.1f}   great-circle={great_circle_km(query, c):7.1f} km")

print("nearest by flat degrees:", min(candidates, key=lambda k: flat_degrees(query, candidates[k])))
print("nearest by great-circle:", min(candidates, key=lambda k: great_circle_km(query, candidates[k])))

The naive distance picks B, but B is 222 km away while A is only 36 km. The flat measure read the dateline jump (+179.5 to -179.5) as 359 degrees apart, when the two points nearly touch. An agent that built a "nearest station" finder this way would return the wrong station near the dateline or the poles, and every test on mid-latitude, mid-ocean data would still pass. The fix is a great-circle distance, or projecting the coordinates first, and the habit is to check that the distance an agent chose actually means what you think on your data.

# Recap

 - A **binary search tree** keeps smaller on the left, larger on the right, so each step discards half.
 - Search cost is the **height** of the tree: `O(log n)` when balanced, `O(n)` when it degenerates on sorted input.
 - Real systems use **balanced trees** and **B-tree indexes** to guarantee `O(log n)` regardless of order.
 - **In-order** traversal yields sorted values (pre- and post-order visit the node at other points); **binary search** on a sorted array is the same halving without a tree.
 - **Removal** has three cases: a leaf, one child, or two children handled through the in-order successor.
 - **kd-trees** extend the idea to space for fast nearest-neighbour search, using whatever distance you give them, so on map coordinates make sure that is a great-circle distance.

**Next time (S19): heaps and sorting** - a tree that lives inside an array, and how the sorts you call every day actually work.